In [2]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import xml.etree.ElementTree as ET

### Import Drugbank

In [3]:
# import xml.etree.ElementTree as ET

# # Load XML
# drugbank_xml = 'Data/DGIDB/drug_bank.xml'
# tree = ET.parse(drugbank_xml)
# root = tree.getroot()

# # Namespace
# ns = {'db': 'http://www.drugbank.ca'}

# Helper to clean tag names
def clean_tag(tag):
    return tag.split('}')[-1] if '}' in tag else tag

# Recursive function to print structure
def print_structure(elem, level=0):
    indent = '  ' * level
    print(f"{indent}- {clean_tag(elem.tag)}")
    for child in elem:
        print_structure(child, level + 1)

# # Get first drug
# first_drug = root.find('db:drug', ns)

# print("🌿 Structure of First Drug Entry:")
# print_structure(first_drug)
# print("\n🌳 Structure of First 3 Drug Entries:")
# drugs = root.findall('db:drug', ns)

# for i, drug in enumerate(drugs[:3]):
#     print(f"\n🔬 Drug {i+1}:")
#     print_structure(drug)


In [4]:
def structure_drug_bank_data(drug_bank_file = 'Data/DGIDB/drug_bank.xml'):
    """
    Function to structure the drug bank data from the XML file.
    :param drug_bank_file: Path to the drug bank XML file.
    :return: DataFrame containing structured drug bank data.
    """
    ### FYI the .find command only finds the first instance of a tag, 
    ### while .findall retrieves all instances of the specified tag within the current element.

    tree = ET.parse(drug_bank_file)
    root = tree.getroot()

    # DrugBank uses a specific namespace
    ns = {'db': 'http://www.drugbank.ca'}
    ### extract all drug elements
    drugs = root.findall('db:drug', ns)
    print(f"Found {len(drugs)} drugs in the DrugBank XML.")
    # Extract drug-gene interactions
    interactions = []
    # The interactions list will store dictionaries with 'drug' and 'gene' keys.
    for drug in root.findall('db:drug', ns): # root.findall('db:drug', ns): Finds all <drug> elements using the namespace.
        drug_name  = drug.find('db:name', ns).text  # drug.find('db:name', ns): Gets the drug's name.
        # print(drug_name)
        for target in drug.findall('db:targets/db:target', ns):  # drug.findall('db:targets/db:target', ns): Finds all <target> elements within <targets>.
            # print(target.tag)
            gene_description = target.find('db:name', ns)  # target.find('db:name', ns): Extracts the gene name for each target.
            poly = target.find('db:polypeptide', ns)  # target.find('db:polypeptide', ns): Extracts the polypeptide information.
            action = target.find('db:actions/db:action', ns) # target.find('db:actions/db:action', ns): Extracts the action of the drug on the target.
            if poly is not None:
                poly_name = poly.find('db:name', ns)
                gene_name = poly.find('db:gene-name', ns)
                specific_function = poly.find('db:specific-function', ns)
                interactions.append({
                    'drug': drug_name,
                    'polypeptide': poly_name.text if poly_name is not None else None,
                    'gene': gene_name.text if gene_name is not None else None,
                    'gene_description': gene_description.text if gene_description is not None else None,
                    'action': action.text if action is not None else None,
                    'specific_function': specific_function.text if specific_function is not None else None
                })
            ############# if polypeptide is not present, we still want to add the drug and gene information
            ############# this is because some drugs may not have a polypeptide associated with them
            ############# but we still want to capture the drug and gene information
            ############# this is common in the DrugBank database, where some drugs target genes directly
            ############# and do not have a polypeptide associated with them

            else:
                gene_name = None
                specific_function = None
                poly_name = None
                action = None
                gene_description = None
                resource = None
                identifier = None
  
                interactions.append({
                        'drug': drug_name,
                        'polypeptide': poly_name.text if poly_name is not None else None,
                        'gene': gene_name.text if gene_name is not None else None,
                        'gene_description': gene_description.text if gene_description is not None else None,
                        'action': action.text if action is not None else None,
                        'specific_function': specific_function.text if specific_function is not None else None
                    })
        
    # Convert to DataFrame
    # Converts the list of dictionaries into a pandas DataFrame, which is easier to analyze, filter, and export.
    df = pd.DataFrame(interactions)

    return df

In [5]:
Drug_bank = structure_drug_bank_data('Data/DGIDB/drug_bank.xml')

Found 17430 drugs in the DrugBank XML.


### Import genetic results

In [6]:
hpv_pos_direct_genes = pd.read_csv('Results/HPV Positive validated genes.csv')
hpv_neg_direct_genes = pd.read_csv('Results/HPV Negative validated genes.csv')


hpv_pos_indirect_genes = pd.read_csv('Results/HPV Positive validated indirect gene results.csv')
hpv_neg_indirect_genes = pd.read_csv('Results/HPV Negative validated indirect gene results.csv')

In [7]:
extracted_target_df= pd.read_csv('../3. Literature based validation/Results/cleaned_extracted_targets_all_pub_after_2000_GPU_2b_gemma.csv')
extracted_target_df_combined = pd.read_csv('../3. Literature based validation/Results/cleaned_extracted_combined_targets_all_pub_after_2000_GPU_2b_gemma.csv')

In [8]:
### connect the direct genes to drug bank
def connect_genes_to_drugbank(gene_df, drugbank_df = Drug_bank):
    connected_drugs = []
    for i, gene in tqdm(enumerate(gene_df['gene_name'])):
        drugs_for_gene = drugbank_df[drugbank_df['gene'] == gene]['drug'].unique().tolist()
        connected_drugs.append(drugs_for_gene)
    gene_df['connected_drugs'] = connected_drugs
    return gene_df

def connect_genes_indirect_to_drugbank(gene_df, drugbank_df = Drug_bank):
    connected_drugs = []
    for i, gene in tqdm(enumerate(gene_df['literature_validated_connected_genes'])):
        indirect_genes = str(gene).split(', ')
        drugs_for_gene = []
        keep_gene = []
        for indirect_gene in indirect_genes:
             drugs = drugbank_df[drugbank_df['gene'] == indirect_gene]['drug'].unique().tolist()
             drugs_for_gene.extend(drugs)
             if len(drugs) > 0:
                 keep_gene.append(indirect_gene)
        gene_df.at[i, 'validated_connected_genes_with_drugbank'] = ', '.join(keep_gene)
        connected_drugs.append(list(set(drugs_for_gene)))  # Remove duplicates
    gene_df['connected_drugs'] = connected_drugs
    gene_df['connected_drugs_count'] = gene_df['connected_drugs'].apply(len)
    return gene_df

In [9]:
# Complete function definition
def final_indirect_gene_Drug_df(gene_df, drugbank_df=Drug_bank, literature_support_df=extracted_target_df):
    """
    Create a comprehensive DataFrame with drug-gene connections including literature support.
    
    Args:
        gene_df: DataFrame with indirect gene results (hpv_pos_indirect_genes or hpv_neg_indirect_genes)
        drugbank_df: DrugBank DataFrame with drug-gene interactions
        literature_support_df: DataFrame with literature support data (extracted_target_df - one row per PMID)
    
    Returns:
        DataFrame with columns:
        - drug: The connected drug
        - connected_gene: The indirect/connected gene that the drug targets
        - root_gene: The root gene that the connected gene relates to
        - literature_support_connected_gene: PMIDs supporting the connected gene
        - literature_support_root_gene: PMIDs supporting the root gene
        - literature_count_connected_gene: Count of PMIDs for connected gene
        - literature_count_root_gene: Count of PMIDs for root gene
        - total_literature_support: Sum of both literature counts
    """
    ### case correct all files in case
    gene_df['gene_name'] = gene_df['gene_name'].str.upper()
    drugbank_df['gene'] = drugbank_df['gene'].str.upper()
    drugbank_df['drug'] = drugbank_df['drug'].str.upper()
    gene_df['literature_validated_connected_genes'] = gene_df['literature_validated_connected_genes'].str.upper()
    gene_df['gene_name'] = gene_df['gene_name'].str.upper()
    literature_support_df['GENE'] = literature_support_df['GENE'].str.upper()


    final_results = []
    
    for i, row in tqdm(gene_df.iterrows(), total=len(gene_df), desc="Processing genes"):
        root_gene = row['gene_name']
        ### add mut_type and q valeu adn empirical q value
        root_gene_mut_type = row.get('MUT_TYPE', 'N/A')  # Get mut_type if it exists, else 'N/A'
        root_gene_q_value = row.get('q_value', 'N/A')  # Get q_value if it exists, else 'N/A'
        root_gene_empirical_q_value = row.get('empirical_q_value', 'N/A')  # Get empirical_q_value if it exists, else 'N/A'
        
        # Get literature support for root gene
        root_literature = literature_support_df[literature_support_df['GENE'].str.upper() == root_gene.upper()]
        root_pmids = sorted(root_literature['PMID'].unique().tolist()) if not root_literature.empty else []
        root_literature_count = len(root_pmids)  # Count unique PMIDs
        
        # Process indirect/connected genes
        if pd.notna(row['literature_validated_connected_genes']):
            indirect_genes = str(row['literature_validated_connected_genes']).split(', ')
            
            for indirect_gene in indirect_genes:
                indirect_gene = indirect_gene.strip()
                
                # Get literature support for connected gene
                connected_literature = literature_support_df[literature_support_df['GENE'].str.upper() == indirect_gene.upper()]
                connected_pmids = sorted(connected_literature['PMID'].unique().tolist()) if not connected_literature.empty else []
                connected_literature_count = len(connected_pmids)  # Count unique PMIDs
                
                # Find drugs connected to this indirect gene
                drugs_for_gene = drugbank_df[drugbank_df['gene'] == indirect_gene]['drug'].unique().tolist()
                
                if drugs_for_gene:  # Only include if there are connected drugs
                    for drug in drugs_for_gene:
                        final_results.append({
                            'drug': drug,
                            'connected_gene': indirect_gene,
                            'root_gene': root_gene,
                            'root_gene_mut_type': root_gene_mut_type,
                            'root_gene_q_value': root_gene_q_value,
                            'root_gene_empirical_q_value': root_gene_empirical_q_value,
                            'literature_support_connected_gene': ', '.join(map(str, connected_pmids)),
                            'literature_support_root_gene': ', '.join(map(str, root_pmids)),
                            'literature_count_connected_gene': connected_literature_count,
                            'literature_count_root_gene': root_literature_count,
                        })
    
    # Convert to DataFrame
    result_df = pd.DataFrame(final_results)
    return result_df

# Usage examples:
# Process HPV positive genes
hpv_pos_final_drug_gene_df = final_indirect_gene_Drug_df(
    hpv_pos_indirect_genes, 
    Drug_bank
)

# Process HPV negative genes
hpv_neg_final_drug_gene_df = final_indirect_gene_Drug_df(
    hpv_neg_indirect_genes, 
    Drug_bank
)

# Display results
print(f"HPV Positive: {len(hpv_pos_final_drug_gene_df)} drug-gene connections")
print(f"HPV Negative: {len(hpv_neg_final_drug_gene_df)} drug-gene connections")

Processing genes: 100%|██████████| 31/31 [00:01<00:00, 16.51it/s]

HPV Positive: 1722 drug-gene connections
HPV Negative: 7516 drug-gene connections


### Import EHR drug results

In [10]:
ehr_hpv_pos_drugs = pd.read_csv('../1. EHR based drug repurposing/Results/ML analysis/hpv_positive_ml_drug_xgb_results.csv')
ehr_hpv_neg_drugs = pd.read_csv('../1. EHR based drug repurposing/Results/ML analysis/hpv_negative_ml_drug_xgb_results.csv')

### replace "_aspirin" in ehr_hpv_neg_drugs with "_acetylsalicylic acid"
ehr_hpv_neg_drugs['feature'] = ehr_hpv_neg_drugs['feature'].str.replace('_aspirin', '_acetylsalicylic acid')

### Combine EHR and genetic results

#### HPV positive

In [11]:
def connect_ehr_and_genetic_indirect(ehr_drug_df, genetic_gene_df):
    ### fix case sensitivity issue
    ehr_drug_df['feature'] = ehr_drug_df['feature'].str.upper()
    genetic_gene_df['drug'] = genetic_gene_df['drug'].str.upper()

    combined_results = []
    for i, row in ehr_drug_df.iterrows():
        drug_name = row['feature'].lstrip('_') # remove the _ prefix
        drug_name = drug_name.upper()  # Normalize to uppercase for matching
        # Check if this drug is connected to any genes in the genetic results
        for j, gene_row in genetic_gene_df.iterrows():
            if drug_name == gene_row['drug']:
                combined_results.append({
                    'ehr_drug': drug_name,
                    'gene_drug': gene_row['drug'],
                    'xgb_importance': row['xgb_importance'],
                    'xgb_absolute_importance': row['xgb_absolute_Importance'],
                    'gene_connected_to': gene_row['connected_gene'],
                    'gene_connected_literature_support': gene_row['literature_support_connected_gene'],
                    'gene_root_gene': gene_row['root_gene'],
                    'gene_root_gene_Mut_type': gene_row.get('root_gene_mut_type', 'N/A'),
                    'gene_root_gene_q_value': gene_row.get('root_gene_q_value', 'N/A'),
                    'gene_root_gene_empirical_q_value': gene_row.get('root_gene_empirical_q_value', 'N/A'),
                    'gene_root_literature_support': gene_row['literature_support_root_gene']
                    # 'genetic_additional_info': gene_row.to_dict()
                })

    overlap_df = pd.DataFrame(combined_results)
    print(f"Number of overlapped medications {len(set(overlap_df['ehr_drug']))}")
    print(f"Out of {len(set(ehr_drug_df['feature']))} medications in EHR data")
    print(f"Overlapped medications are: {list(set(overlap_df['ehr_drug']))}")
    return overlap_df

In [12]:
hpv_pos_validated_indirect = connect_ehr_and_genetic_indirect(ehr_hpv_pos_drugs, hpv_pos_final_drug_gene_df)

Number of overlapped medications 3
Out of 23 medications in EHR data
Overlapped medications are: ['DEXAMETHASONE', 'LEVOTHYROXINE', 'HEPARIN']


In [13]:
hpv_pos_validated_indirect

,ehr_drug,gene_drug,xgb_importance,xgb_absolute_importance,gene_connected_to,gene_connected_literature_support,gene_root_gene,gene_root_gene_Mut_type,gene_root_gene_q_value,gene_root_gene_empirical_q_value,gene_root_literature_support
0,LEVOTHYROXINE,LEVOTHYROXINE,0.044148,0.044148,THRB,17210745,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301","11358835, 11836556, 11959846, 14581353, 155436..."
1,HEPARIN,HEPARIN,0.009594,0.009594,FGFR2,"12075207, 12569015",PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301","11358835, 11836556, 11959846, 14581353, 155436..."
2,HEPARIN,HEPARIN,0.009594,0.009594,FGFR1,"11562460, 15165306, 16807070, 17545628, 18059337",PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301","11358835, 11836556, 11959846, 14581353, 155436..."
3,HEPARIN,HEPARIN,0.009594,0.009594,FGFR4,11562460,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301","11358835, 11836556, 11959846, 14581353, 155436..."
4,HEPARIN,HEPARIN,0.009594,0.009594,FGF4,"12429977, 15942670, 17016592, 17487429, 18405350",SOX2,AMPLIFICATION,1.2920437544143294e-55,0.0115776022868357,15942670
5,DEXAMETHASONE,DEXAMETHASONE,0.008129,0.008129,ANXA1,"16270603, 16627980, 16899607",SOX2,AMPLIFICATION,1.2920437544143294e-55,0.0115776022868357,15942670


In [14]:
hpv_pos_validated_indirect.to_csv('Results/HPV positive EHR drug candidates validated indirect.csv', index=False)

In [15]:
hpv_pos_validated_indirect['ehr_drug'].unique().tolist()

['LEVOTHYROXINE', 'HEPARIN', 'DEXAMETHASONE']

#### HPV negative

In [16]:
hpv_neg_final_drug_gene_df

,drug,connected_gene,root_gene,root_gene_mut_type,root_gene_q_value,root_gene_empirical_q_value,literature_support_connected_gene,literature_support_root_gene,literature_count_connected_gene,literature_count_root_gene
0,ACITRETIN,STAT3,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,"11222718, 11426647, 11773428, 12067972, 121624...","11224600, 11420458, 14685876, 17429099",41,4
1,CELECOXIB,STAT3,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,"11222718, 11426647, 11773428, 12067972, 121624...","11224600, 11420458, 14685876, 17429099",41,4
2,QUERCETIN,STAT3,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,"11222718, 11426647, 11773428, 12067972, 121624...","11224600, 11420458, 14685876, 17429099",41,4
3,GOLOTIMOD,STAT3,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,"11222718, 11426647, 11773428, 12067972, 121624...","11224600, 11420458, 14685876, 17429099",41,4
4,ATIPRIMOD,STAT3,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,"11222718, 11426647, 11773428, 12067972, 121624...","11224600, 11420458, 14685876, 17429099",41,4
...,...,...,...,...,...,...,...,...,...,...
7511,LIFITEGRAST,ICAM1,TP53,SOMATIC,0.0,0.005960073448722,16309178,"11390535, 11445847, 11445859, 11916556, 125898...",1,29
7512,NAFAMOSTAT,ICAM1,TP53,SOMATIC,0.0,0.005960073448722,16309178,"11390535, 11445847, 11445859, 11916556, 125898...",1,29
7513,"1,2,3-TRIHYDROXY-1,2,3,4-TETRAHYDROBENZO[A]PYRENE",POLK,TP53,SOMATIC,0.0,0.005960073448722,17494052,"11390535, 11445847, 11445859, 11916556, 125898...",1,29
7514,"1-[2-DEOXYRIBOFURANOSYL]-2,4-DIFLUORO-5-METHYL...",POLK,TP53,SOMATIC,0.0,0.005960073448722,17494052,"11390535, 11445847, 11445859, 11916556, 125898...",1,29


In [17]:
hpv_neg_validated_indirect = connect_ehr_and_genetic_indirect(ehr_hpv_neg_drugs, hpv_neg_final_drug_gene_df)

Number of overlapped medications 5
Out of 36 medications in EHR data
Overlapped medications are: ['LEVOTHYROXINE', 'METHYLPREDNISOLONE', 'MELATONIN', 'ACETYLSALICYLIC ACID', 'ACETAMINOPHEN']


In [18]:
hpv_neg_validated_indirect

,ehr_drug,gene_drug,xgb_importance,xgb_absolute_importance,gene_connected_to,gene_connected_literature_support,gene_root_gene,gene_root_gene_Mut_type,gene_root_gene_q_value,gene_root_gene_empirical_q_value,gene_root_literature_support
0,MELATONIN,MELATONIN,0.010572,0.010572,ESR1,11309301,HRAS,SOMATIC,1.4855655450622938e-26,0.005960073448722,16676365
1,MELATONIN,MELATONIN,0.010572,0.010572,ESR1,11309301,PIK3CA,"AMPLIFICATION, SOMATIC","1.5505691778987169e-208, 8.841194236005348e-40","0.0043623451388343, 0.005960073448722","11358835, 11836556, 11959846, 14581353, 155436..."
2,MELATONIN,MELATONIN,0.010572,0.010572,ESR1,11309301,TP53,SOMATIC,0.0,0.005960073448722,"11390535, 11445847, 11445859, 11916556, 125898..."
3,LEVOTHYROXINE,LEVOTHYROXINE,0.004997,0.004997,THRB,17210745,PIK3CA,"AMPLIFICATION, SOMATIC","1.5505691778987169e-208, 8.841194236005348e-40","0.0043623451388343, 0.005960073448722","11358835, 11836556, 11959846, 14581353, 155436..."
4,LEVOTHYROXINE,LEVOTHYROXINE,0.004997,0.004997,ITGAV,18427826,RHOA,SOMATIC,0.0051149638902022,0.005960073448722,"12082550, 16369533, 17234718"
5,LEVOTHYROXINE,LEVOTHYROXINE,0.004997,0.004997,ITGB3,17679464,RHOA,SOMATIC,0.0051149638902022,0.005960073448722,"12082550, 16369533, 17234718"
6,LEVOTHYROXINE,LEVOTHYROXINE,0.004997,0.004997,ITGB3,17679464,SELP,SOMATIC,0.0366377654726179,0.005960073448722,16135921
7,LEVOTHYROXINE,LEVOTHYROXINE,0.004997,0.004997,THRB,17210745,TP53,SOMATIC,0.0,0.005960073448722,"11390535, 11445847, 11445859, 11916556, 125898..."
8,ACETAMINOPHEN,ACETAMINOPHEN,0.004543,0.004543,PTGS2,18030354,CASP8,SOMATIC,1.4328145167446222e-45,0.005960073448722,16857411
9,ACETAMINOPHEN,ACETAMINOPHEN,0.004543,0.004543,PTGS2,18030354,TP53,SOMATIC,0.0,0.005960073448722,"11390535, 11445847, 11445859, 11916556, 125898..."


In [19]:
hpv_neg_validated_indirect[hpv_neg_validated_indirect['ehr_drug'] != 'ACETYLsalicylic acid'.upper()]

,ehr_drug,gene_drug,xgb_importance,xgb_absolute_importance,gene_connected_to,gene_connected_literature_support,gene_root_gene,gene_root_gene_Mut_type,gene_root_gene_q_value,gene_root_gene_empirical_q_value,gene_root_literature_support
0,MELATONIN,MELATONIN,0.010572,0.010572,ESR1,11309301,HRAS,SOMATIC,1.4855655450622938e-26,0.005960073448722,16676365
1,MELATONIN,MELATONIN,0.010572,0.010572,ESR1,11309301,PIK3CA,"AMPLIFICATION, SOMATIC","1.5505691778987169e-208, 8.841194236005348e-40","0.0043623451388343, 0.005960073448722","11358835, 11836556, 11959846, 14581353, 155436..."
2,MELATONIN,MELATONIN,0.010572,0.010572,ESR1,11309301,TP53,SOMATIC,0.0,0.005960073448722,"11390535, 11445847, 11445859, 11916556, 125898..."
3,LEVOTHYROXINE,LEVOTHYROXINE,0.004997,0.004997,THRB,17210745,PIK3CA,"AMPLIFICATION, SOMATIC","1.5505691778987169e-208, 8.841194236005348e-40","0.0043623451388343, 0.005960073448722","11358835, 11836556, 11959846, 14581353, 155436..."
4,LEVOTHYROXINE,LEVOTHYROXINE,0.004997,0.004997,ITGAV,18427826,RHOA,SOMATIC,0.0051149638902022,0.005960073448722,"12082550, 16369533, 17234718"
5,LEVOTHYROXINE,LEVOTHYROXINE,0.004997,0.004997,ITGB3,17679464,RHOA,SOMATIC,0.0051149638902022,0.005960073448722,"12082550, 16369533, 17234718"
6,LEVOTHYROXINE,LEVOTHYROXINE,0.004997,0.004997,ITGB3,17679464,SELP,SOMATIC,0.0366377654726179,0.005960073448722,16135921
7,LEVOTHYROXINE,LEVOTHYROXINE,0.004997,0.004997,THRB,17210745,TP53,SOMATIC,0.0,0.005960073448722,"11390535, 11445847, 11445859, 11916556, 125898..."
8,ACETAMINOPHEN,ACETAMINOPHEN,0.004543,0.004543,PTGS2,18030354,CASP8,SOMATIC,1.4328145167446222e-45,0.005960073448722,16857411
9,ACETAMINOPHEN,ACETAMINOPHEN,0.004543,0.004543,PTGS2,18030354,TP53,SOMATIC,0.0,0.005960073448722,"11390535, 11445847, 11445859, 11916556, 125898..."


In [20]:
hpv_neg_validated_indirect.to_csv('Results/HPV negative EHR drug candidates validated indirect.csv', index=False)

In [21]:
hpv_neg_validated_indirect['ehr_drug'].unique().tolist()

['MELATONIN',
 'LEVOTHYROXINE',
 'ACETAMINOPHEN',
 'METHYLPREDNISOLONE',
 'ACETYLSALICYLIC ACID']

### display top direct genes connected to EHR drugs

In [22]:
hpv_pos_validated_indirect

,ehr_drug,gene_drug,xgb_importance,xgb_absolute_importance,gene_connected_to,gene_connected_literature_support,gene_root_gene,gene_root_gene_Mut_type,gene_root_gene_q_value,gene_root_gene_empirical_q_value,gene_root_literature_support
0,LEVOTHYROXINE,LEVOTHYROXINE,0.044148,0.044148,THRB,17210745,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301","11358835, 11836556, 11959846, 14581353, 155436..."
1,HEPARIN,HEPARIN,0.009594,0.009594,FGFR2,"12075207, 12569015",PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301","11358835, 11836556, 11959846, 14581353, 155436..."
2,HEPARIN,HEPARIN,0.009594,0.009594,FGFR1,"11562460, 15165306, 16807070, 17545628, 18059337",PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301","11358835, 11836556, 11959846, 14581353, 155436..."
3,HEPARIN,HEPARIN,0.009594,0.009594,FGFR4,11562460,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301","11358835, 11836556, 11959846, 14581353, 155436..."
4,HEPARIN,HEPARIN,0.009594,0.009594,FGF4,"12429977, 15942670, 17016592, 17487429, 18405350",SOX2,AMPLIFICATION,1.2920437544143294e-55,0.0115776022868357,15942670
5,DEXAMETHASONE,DEXAMETHASONE,0.008129,0.008129,ANXA1,"16270603, 16627980, 16899607",SOX2,AMPLIFICATION,1.2920437544143294e-55,0.0115776022868357,15942670


In [23]:
### display genetic resutls of direct genes connected to EHR drugs
hpv_pos_validated_indirect[['ehr_drug', 'xgb_importance', 'xgb_absolute_importance', 'gene_connected_to', 'gene_root_gene']]
root_genes_hpv_pos = hpv_pos_validated_indirect['gene_root_gene'].unique().tolist()
root_genes_hpv_pos
hpv_pos_direct_genes[hpv_pos_direct_genes['gene_name'].isin(root_genes_hpv_pos)]

,Unnamed: 0,gene_name,MUT_TYPE,q_value,empirical_q_value,GENE,PMID,NUMBER_OF_ARTICLES
4,4,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301",PIK3CA,"11358835, 11836556, 11959846, 14581353, 155436...",17.0
6,6,SOX2,AMPLIFICATION,1.2920437544143294e-55,0.0115776022868357,SOX2,15942670,1.0


In [24]:
### display genetic resutls of direct genes connected to EHR drugs
root_genes_hpv_neg = hpv_neg_validated_indirect['gene_root_gene'].unique().tolist()
root_genes_hpv_neg
hpv_neg_direct_genes[hpv_neg_direct_genes['gene_name'].isin(root_genes_hpv_neg)]

,Unnamed: 0,gene_name,MUT_TYPE,q_value,empirical_q_value,GENE,PMID,NUMBER_OF_ARTICLES
1,1,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,BCL6,"11224600, 11420458, 14685876, 17429099",4.0
2,2,CASP8,SOMATIC,1.4328145167446222e-45,0.005960073448722,CASP8,16857411,1.0
4,4,CDKN2A,"DELETION, SOMATIC","0.0, 2.518854531227791e-131","0.0191774659792392, 0.005960073448722",CDKN2A,"11309301, 11720478, 12459644, 12632081, 126844...",16.0
5,5,CDKN2B,DELETION,0.0,0.0191774659792392,CDKN2B,"12632081, 17673925",2.0
13,13,HLA-A,SOMATIC,3.134055903831376e-07,0.005960073448722,HLA-A,11577000,1.0
14,14,HRAS,SOMATIC,1.4855655450622938e-26,0.005960073448722,HRAS,16676365,1.0
15,15,KEAP1,SOMATIC,3.916198590432336e-05,0.005960073448722,KEAP1,16637057,1.0
19,19,NOTCH1,SOMATIC,5.625692605377536e-34,0.005960073448722,NOTCH1,"12539049, 16219138",2.0
21,21,PIK3CA,"AMPLIFICATION, SOMATIC","1.5505691778987169e-208, 8.841194236005348e-40","0.0043623451388343, 0.005960073448722",PIK3CA,"11358835, 11836556, 11959846, 14581353, 155436...",17.0
25,25,RFC4,AMPLIFICATION,8.446185897603275e-199,0.0043623451388343,RFC4,16467079,1.0


In [25]:
def restructure_ehr_validated_indirect(ehr_validated_df, indirect_genes_df, literature_support_df=extracted_target_df):
    """
    Restructure EHR validated indirect results to properly show each connected gene with its information.
    
    Args:
        ehr_validated_df: The validated EHR-genetic results (hpv_pos_validated_indirect or hpv_neg_validated_indirect)
        indirect_genes_df: The original indirect genes dataframe (hpv_pos_indirect_genes or hpv_neg_indirect_genes)
        literature_support_df: DataFrame with literature support data (extracted_target_df - one row per PMID)
    
    Returns:
        DataFrame with one row per connected gene showing:
        - connected_gene: The indirect gene
        - ehr_drugs_targeting_gene: List of EHR drugs that target this gene
        - literature_support_connected_gene: PMIDs supporting the connected gene
        - literature_count_connected_gene: Count of PMIDs for connected gene
        - root_gene: The mutated root gene
        - MUT_TYPE: Mutation type
        - q_value: q-value
        - empirical_q_value: Empirical q-value
        - literature_support_root_gene: PMIDs supporting root gene
        - literature_count_root_gene: Count of PMIDs for root gene
        - xgb_importance_list: List of XGB importance values for each drug
        - total_literature_support: Sum of literature counts
    """
    
    # Group by connected gene and root gene to aggregate drugs
    results = []
    
    # Get unique combinations of connected gene and root gene
    unique_combinations = ehr_validated_df.groupby(['gene_connected_to', 'gene_root_gene'])
    
    for (connected_gene, root_gene), group in tqdm(unique_combinations, desc="Processing gene combinations"):
        # Get all EHR drugs targeting this connected gene
        ehr_drugs = group['ehr_drug'].unique().tolist()
        xgb_importance_list = group['xgb_importance'].tolist()
        xgb_absolute_importance_list = group['xgb_absolute_importance'].tolist()
        
        # Get literature support for connected gene
        connected_literature = literature_support_df[literature_support_df['GENE'].str.upper() == connected_gene.upper()]
        connected_pmids = sorted(connected_literature['PMID'].unique().tolist()) if not connected_literature.empty else []
        connected_literature_count = len(connected_pmids)
        
        # Get root gene information from the indirect_genes_df
        root_gene_row = indirect_genes_df[indirect_genes_df['gene_name'] == root_gene]
        
        if not root_gene_row.empty:
            root_gene_row = root_gene_row.iloc[0]
            
            # Get literature support for root gene
            root_literature = literature_support_df[literature_support_df['GENE'].str.upper() == root_gene.upper()]
            root_pmids = sorted(root_literature['PMID'].unique().tolist()) if not root_literature.empty else []
            root_literature_count = len(root_pmids)
            
            results.append({
                'connected_gene': connected_gene,
                'ehr_drugs_targeting_gene': ', '.join(ehr_drugs),
                'num_ehr_drugs': len(ehr_drugs),
                'xgb_importance_values': ', '.join([f"{val:.4f}" for val in xgb_importance_list]),
                'xgb_absolute_importance_values': ', '.join([f"{val:.4f}" for val in xgb_absolute_importance_list]),
                'mean_xgb_importance': np.mean(xgb_importance_list),
                'literature_support_connected_gene': ', '.join(map(str, connected_pmids) ),
                'literature_count_connected_gene': connected_literature_count,
                'root_gene': root_gene,
                'MUT_TYPE': root_gene_row['MUT_TYPE'],
                'q_value': root_gene_row['q_value'],
                'empirical_q_value': root_gene_row['empirical_q_value'],
                'literature_support_root_gene': ', '.join(map(str, root_pmids)),
                'literature_count_root_gene': root_literature_count,
                'root_gene_PMID': root_gene_row['PMID'] if 'PMID' in root_gene_row else None,
                'root_gene_NUMBER_OF_ARTICLES': root_gene_row['NUMBER_OF_ARTICLES'] if 'NUMBER_OF_ARTICLES' in root_gene_row else None,
            })
    
    # Convert to DataFrame
    result_df = pd.DataFrame(results)
    return result_df

# Process HPV positive validated results
print("Processing HPV positive validated indirect results...")
hpv_pos_restructured_indirect = restructure_ehr_validated_indirect(
    hpv_pos_validated_indirect,
    hpv_pos_indirect_genes,
    extracted_target_df
)

# Process HPV negative validated results
print("Processing HPV negative validated indirect results...")
hpv_neg_restructured_indirect = restructure_ehr_validated_indirect(
    hpv_neg_validated_indirect,
    hpv_neg_indirect_genes,
    extracted_target_df
)

# Display results
print(f"\nHPV Positive: {len(hpv_pos_restructured_indirect)} unique connected genes")
print(f"HPV Negative: {len(hpv_neg_restructured_indirect)} unique connected genes")

Processing HPV positive validated indirect results...


Processing gene combinations: 100%|██████████| 6/6 [00:00<00:00, 258.97it/s]


Processing HPV negative validated indirect results...


Processing gene combinations: 100%|██████████| 43/43 [00:00<00:00, 281.13it/s]


HPV Positive: 6 unique connected genes
HPV Negative: 43 unique connected genes


In [26]:
hpv_pos_restructured_indirect

,connected_gene,ehr_drugs_targeting_gene,num_ehr_drugs,xgb_importance_values,xgb_absolute_importance_values,mean_xgb_importance,literature_support_connected_gene,literature_count_connected_gene,root_gene,MUT_TYPE,q_value,empirical_q_value,literature_support_root_gene,literature_count_root_gene,root_gene_PMID,root_gene_NUMBER_OF_ARTICLES
0,ANXA1,DEXAMETHASONE,1,0.0081,0.0081,0.008129,"16270603, 16627980, 16899607",3,SOX2,AMPLIFICATION,1.2920437544143294e-55,0.0115776022868357,15942670,1,15942670,1.0
1,FGF4,HEPARIN,1,0.0096,0.0096,0.009594,"12429977, 15942670, 17016592, 17487429, 18405350",5,SOX2,AMPLIFICATION,1.2920437544143294e-55,0.0115776022868357,15942670,1,15942670,1.0
2,FGFR1,HEPARIN,1,0.0096,0.0096,0.009594,"11562460, 15165306, 16807070, 17545628, 18059337",5,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301","11358835, 11836556, 11959846, 14581353, 155436...",17,"11358835, 11836556, 11959846, 14581353, 155436...",17.0
3,FGFR2,HEPARIN,1,0.0096,0.0096,0.009594,"12075207, 12569015",2,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301","11358835, 11836556, 11959846, 14581353, 155436...",17,"11358835, 11836556, 11959846, 14581353, 155436...",17.0
4,FGFR4,HEPARIN,1,0.0096,0.0096,0.009594,11562460,1,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301","11358835, 11836556, 11959846, 14581353, 155436...",17,"11358835, 11836556, 11959846, 14581353, 155436...",17.0
5,THRB,LEVOTHYROXINE,1,0.0441,0.0441,0.044148,17210745,1,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301","11358835, 11836556, 11959846, 14581353, 155436...",17,"11358835, 11836556, 11959846, 14581353, 155436...",17.0


In [27]:
hpv_neg_restructured_indirect[~((hpv_neg_restructured_indirect['ehr_drugs_targeting_gene'] == 'ACETYLSALICYLIC ACID') & (hpv_neg_restructured_indirect['connected_gene'] == 'TP53'))].sort_values(by='ehr_drugs_targeting_gene', ascending=True)[['ehr_drugs_targeting_gene', 'xgb_importance_values', 'connected_gene', 'literature_support_connected_gene', 'root_gene', 'MUT_TYPE', 'q_value', 'empirical_q_value', 'literature_support_root_gene']]

,ehr_drugs_targeting_gene,xgb_importance_values,connected_gene,literature_support_connected_gene,root_gene,MUT_TYPE,q_value,empirical_q_value,literature_support_root_gene
30,"ACETAMINOPHEN, ACETYLSALICYLIC ACID","0.0045, 0.0033",PTGS2,18030354,TP53,SOMATIC,0.0,0.005960073448722,"11390535, 11445847, 11445859, 11916556, 125898..."
29,"ACETAMINOPHEN, ACETYLSALICYLIC ACID","0.0045, 0.0033",PTGS2,18030354,CASP8,SOMATIC,1.4328145167446222e-45,0.005960073448722,16857411
16,ACETYLSALICYLIC ACID,0.0033,MYC,"11368094, 11911252, 14499691, 16807070, 18000363",BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,"11224600, 11420458, 14685876, 17429099"
20,ACETYLSALICYLIC ACID,0.0033,MYC,"11368094, 11911252, 14499691, 16807070, 18000363",HRAS,SOMATIC,1.4855655450622938e-26,0.005960073448722,16676365
19,ACETYLSALICYLIC ACID,0.0033,MYC,"11368094, 11911252, 14499691, 16807070, 18000363",CDKN2B,DELETION,0.0,0.0191774659792392,"12632081, 17673925"
18,ACETYLSALICYLIC ACID,0.0033,MYC,"11368094, 11911252, 14499691, 16807070, 18000363",CDKN2A,"DELETION, SOMATIC","0.0, 2.518854531227791e-131","0.0191774659792392, 0.005960073448722","11309301, 11720478, 12459644, 12632081, 126844..."
17,ACETYLSALICYLIC ACID,0.0033,MYC,"11368094, 11911252, 14499691, 16807070, 18000363",CASP8,SOMATIC,1.4328145167446222e-45,0.005960073448722,16857411
23,ACETYLSALICYLIC ACID,0.0033,MYC,"11368094, 11911252, 14499691, 16807070, 18000363",RHOA,SOMATIC,0.0051149638902022,0.005960073448722,"12082550, 16369533, 17234718"
24,ACETYLSALICYLIC ACID,0.0033,MYC,"11368094, 11911252, 14499691, 16807070, 18000363",SOX2,AMPLIFICATION,1.960361075687212e-205,0.0043623451388343,15942670
25,ACETYLSALICYLIC ACID,0.0033,MYC,"11368094, 11911252, 14499691, 16807070, 18000363",TP53,SOMATIC,0.0,0.005960073448722,"11390535, 11445847, 11445859, 11916556, 125898..."


In [28]:
hpv_neg_restructured_indirect[hpv_neg_restructured_indirect['ehr_drugs_targeting_gene'] == 'ACETYLSALICYLIC ACID']

,connected_gene,ehr_drugs_targeting_gene,num_ehr_drugs,xgb_importance_values,xgb_absolute_importance_values,mean_xgb_importance,literature_support_connected_gene,literature_count_connected_gene,root_gene,MUT_TYPE,q_value,empirical_q_value,literature_support_root_gene,literature_count_root_gene,root_gene_PMID,root_gene_NUMBER_OF_ARTICLES
2,CCND1,ACETYLSALICYLIC ACID,1,0.0033,0.0033,0.003261,"11470749, 11859213, 11958128, 12649172, 144996...",25,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,"11224600, 11420458, 14685876, 17429099",4,"11224600, 11420458, 14685876, 17429099",4.0
3,CCND1,ACETYLSALICYLIC ACID,1,0.0033,0.0033,0.003261,"11470749, 11859213, 11958128, 12649172, 144996...",25,CDKN2A,"DELETION, SOMATIC","0.0, 2.518854531227791e-131","0.0191774659792392, 0.005960073448722","11309301, 11720478, 12459644, 12632081, 126844...",16,"11309301, 11720478, 12459644, 12632081, 126844...",16.0
4,CCND1,ACETYLSALICYLIC ACID,1,0.0033,0.0033,0.003261,"11470749, 11859213, 11958128, 12649172, 144996...",25,CDKN2B,DELETION,0.0,0.0191774659792392,"12632081, 17673925",2,"12632081, 17673925",2.0
5,CCND1,ACETYLSALICYLIC ACID,1,0.0033,0.0033,0.003261,"11470749, 11859213, 11958128, 12649172, 144996...",25,HRAS,SOMATIC,1.4855655450622938e-26,0.005960073448722,16676365,1,16676365,1.0
6,CCND1,ACETYLSALICYLIC ACID,1,0.0033,0.0033,0.003261,"11470749, 11859213, 11958128, 12649172, 144996...",25,NOTCH1,SOMATIC,5.625692605377536e-34,0.005960073448722,"12539049, 16219138",2,"12539049, 16219138",2.0
7,CCND1,ACETYLSALICYLIC ACID,1,0.0033,0.0033,0.003261,"11470749, 11859213, 11958128, 12649172, 144996...",25,PIK3CA,"AMPLIFICATION, SOMATIC","1.5505691778987169e-208, 8.841194236005348e-40","0.0043623451388343, 0.005960073448722","11358835, 11836556, 11959846, 14581353, 155436...",17,"11358835, 11836556, 11959846, 14581353, 155436...",17.0
8,CCND1,ACETYLSALICYLIC ACID,1,0.0033,0.0033,0.003261,"11470749, 11859213, 11958128, 12649172, 144996...",25,SOX2,AMPLIFICATION,1.960361075687212e-205,0.0043623451388343,15942670,1,15942670,1.0
9,CCND1,ACETYLSALICYLIC ACID,1,0.0033,0.0033,0.003261,"11470749, 11859213, 11958128, 12649172, 144996...",25,TP53,SOMATIC,0.0,0.005960073448722,"11390535, 11445847, 11445859, 11916556, 125898...",29,"11390535, 11445847, 11445859, 11916556, 125898...",29.0
16,MYC,ACETYLSALICYLIC ACID,1,0.0033,0.0033,0.003261,"11368094, 11911252, 14499691, 16807070, 18000363",5,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,"11224600, 11420458, 14685876, 17429099",4,"11224600, 11420458, 14685876, 17429099",4.0
17,MYC,ACETYLSALICYLIC ACID,1,0.0033,0.0033,0.003261,"11368094, 11911252, 14499691, 16807070, 18000363",5,CASP8,SOMATIC,1.4328145167446222e-45,0.005960073448722,16857411,1,16857411,1.0


##### Sanity check

In [29]:
Drug_bank[(Drug_bank['drug']=='LEVOTHYROXINE') & (Drug_bank['gene']=='THRB')]

,drug,polypeptide,gene,gene_description,action,specific_function
2275,LEVOTHYROXINE,Thyroid hormone receptor beta,THRB,Thyroid hormone receptor beta,agonist,chromatin DNA binding
